<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/Euler-equations-compressible-flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Euler equations for compressible flow** **Johan Hoffman**

# **Abstract**

This short report shows an example on how to use FEniCSx to solve the Euler equations for compressible flow. A stabilized finite element method with shock-capturing diffusion is applied to simulate supersonic flow past a circular cylinder.

In [ ]:
# Copyright (C) 2019-2026 Johan Hoffman (jhoffman@kth.se)
#
# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This template is maintained by Johan Hoffman
# Please report problems to jhoffman@kth.se

# Setup environment

In [ ]:
# --- canonical: bootstrap v2 ---
import os as _os
_os.environ.setdefault("OMP_NUM_THREADS", "1")
_os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
_os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
_os.environ.setdefault("MKL_NUM_THREADS", "1")

import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.
    dist_min : float or None
        Threshold DistMin per hole (inner boundary of transition zone).
        Default None → 0.0 (fine zone starts at the polygon surface).
    dist_max : float or None
        Threshold DistMax per hole (outer boundary of transition zone).
        Default None → r/2 per hole (reproduces v5 behaviour exactly).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance
    ``dist_max`` (default r/2).  No MeshSizeMax override; background field
    is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at dist_max: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v1 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

In [ ]:
# --- canonical: time_step v1 ---
import numpy as np


def time_step(msh, U_ref, C_CFL=0.5):
    """Return C_CFL * h_min / U_ref (CFL-based time step)."""
    tdim = msh.topology.dim
    n_cells = msh.topology.index_map(tdim).size_local
    h_min = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
    return C_CFL * h_min / U_ref
# --- end canonical: time_step ---

In [ ]:
import numpy as np
import time
import math
from mpi4py import MPI
from dolfinx.fem import (
    functionspace, Function, Constant, form,
    assemble_scalar, locate_dofs_topological, dirichletbc,
    Expression,
)
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc
import ufl
import basix.ufl as bufl
import matplotlib.pyplot as plt

# **Introduction**

For density $\rho$, momentum $m$, and total energy $E$, the Euler equations take the form

$$\frac{\partial \rho}{\partial t} + \nabla \cdot (\rho u) = 0$$

$$\frac{\partial m}{\partial t} + \nabla \cdot (m \otimes u) + \nabla p = 0$$

$$\frac{\partial E}{\partial t} + \nabla \cdot (Eu + pu) = 0$$

where $u = m/\rho$ is the velocity and $p = (\gamma-1)\bigl(E - \tfrac{1}{2}|m|^2/\rho\bigr)$ is the pressure.

# **Method**

**Define domain and mesh**

In [ ]:
# Define length scale
L_ref = 1.0

# Define rectangular domain
L = 5.0
H = 4.0

# Model diameter
MD = 0.4

# Define circle
xc = 1.5
yc = 0.5*H
rc = 0.5*MD

# Generate mesh
resolution = 64
msh, _ = gmsh_rect_minus_circles(L, H, [(xc, yc, rc)], resolution)
facet_tags = tag_boundaries(msh, L, H)

# Local mesh refinement (no_levels=0; increase for finer boundary layers)
no_levels = 0
for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda midpoints: (np.hypot(midpoints[:, 0] - xc, midpoints[:, 1] - yc) < 1.0),
    )
    facet_tags = tag_boundaries(msh, L, H)

tdim   = msh.topology.dim
fdim   = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local

plot_mesh(msh)

**Define physics model**

In [ ]:
# Adiabatic index gamma = cp/cv (gamma = 1.4 for air)
gamma = 1.4

# Specific gas constant [J/(kg*K)]
R_constant = 287

# Specific heat at constant volume [J/(kg*K)]
cv = R_constant / (gamma - 1.0)

# Free stream flow data (used as boundary data)
r_fs    = 1.225  # [kg/m3]
Temp_fs = 290    # [K]
u_fs    = 500    # [m/s]
print('Free stream density [kg/m3]:', r_fs)
print('Free stream temperature [K]:', Temp_fs)
print('Free stream velocity [m/s]:', u_fs)

m_fs = u_fs * r_fs
c_fs = math.sqrt(gamma * R_constant * Temp_fs)
M_fs = u_fs / c_fs
print('Mach number (free stream):', M_fs)

e_fs = cv * Temp_fs
E_fs = r_fs * (e_fs + 0.5 * u_fs * u_fs)

# Canonical nondimensional form – reference normalizations
r_ref = 1.225   # [kg/m3] – sea-level density
u_ref = 340.3   # [m/s]  – sea-level speed of sound
m_ref = r_ref * u_ref
E_ref = r_ref * u_ref * u_ref
p_ref = r_ref * u_ref * u_ref

p_fs = r_fs * Temp_fs * R_constant
print('Free stream pressure [Pa]:', p_fs)
print('Free stream pressure (nondimensional):', p_fs / p_ref)

T_ref = L_ref / u_ref
print('Reference time [s]:', T_ref)

**Define finite element approximation spaces**

In [ ]:
# Generate finite element spaces (for density, momentum, total energy)
P1s = bufl.element('Lagrange', msh.basix_cell(), 1)
P1v = bufl.element('Lagrange', msh.basix_cell(), 1, shape=(2,))

R = functionspace(msh, P1s)   # density
V = functionspace(msh, P1v)   # momentum
Q = functionspace(msh, P1s)   # total energy

# Define trial and test functions
r = ufl.TrialFunction(R)
m = ufl.TrialFunction(V)
E = ufl.TrialFunction(Q)
d = ufl.TestFunction(R)
v = ufl.TestFunction(V)
q = ufl.TestFunction(Q)

**Define boundary conditions**

In [ ]:
# Nondimensional inflow values
r_in     = r_fs / r_ref
m_in     = m_fs / m_ref
E_in     = E_fs / E_ref
m_in_vec = ufl.as_vector([m_in, 0.0])

def _bc(space, value, tag):
    dofs = locate_dofs_topological(space, fdim, facet_tags.find(tag))
    return dirichletbc(PETSc.ScalarType(value), dofs, space)

bcm_in0  = _bc(V.sub(0), m_in, 1)   # m_x = m_in at inflow
bcm_in1  = _bc(V.sub(1), 0.0,  1)   # m_y = 0    at inflow
bcm_upp1 = _bc(V.sub(1), 0.0,  4)   # m_y = 0    at upper wall (slip)
bcm_low1 = _bc(V.sub(1), 0.0,  3)   # m_y = 0    at lower wall (slip)
#bcm_obj0 = _bc(V.sub(0), 0.0, 5)   # cylinder no-slip (commented as in legacy)
#bcm_obj1 = _bc(V.sub(1), 0.0, 5)

# Slip boundary condition (commented out as in legacy):
#V_slip = FunctionSpace(mesh, 'CG', 1)
#bcm_slip_0 = Function(V_slip); bcm_slip_1 = Function(V_slip)

# Inflow BC based on characteristics: supersonic -> all components fixed
if M_fs > 1.0:
    bcm = [bcm_in0, bcm_in1, bcm_upp1, bcm_low1]
else:
    bcm = [bcm_in0, bcm_upp1, bcm_low1]

bcr = [_bc(R, r_in, 1)]
bcE = [_bc(Q, E_in, 1)]

# Approximate facet normal (commented out as in legacy):
#n = ufl.FacetNormal(msh)
#V_normal = functionspace(msh, P1v)

# Boundary measure
ds_m = ufl.Measure('ds', domain=msh, subdomain_data=facet_tags)

# **Results**

**Define flow parameters**

In [ ]:
# Length of time interval [nondimensional]
T = 10.0
plot_freq = 20

**Define method parameters**

In [ ]:
# Define iteration functions
# *0 solution from previous time step
# *1 linearized solution at present time step
r0 = Function(R, name='r0'); r1 = Function(R, name='r1')
m0 = Function(V, name='m0'); m1 = Function(V, name='m1')
E0 = Function(Q, name='E0'); E1 = Function(Q, name='E1')

# Lower threshold for density (cavitation fix)
eps_cav = 1.0e-6

# Set parameters for nonlinear and linear solvers
num_nnlin_iter = 5

# Time step length (CFL-based): dt = hmin / (u_in + c_in)
u_in = m_in / r_in
c_in = math.sqrt(gamma * (gamma - 1.0) * (E_in / r_in - 0.5 * u_in * u_in))
dt   = time_step(msh, u_in + c_in, C_CFL=1.0)
dt_c = Constant(msh, PETSc.ScalarType(dt))

print(f'u_in = {u_in:.4f}  c_in = {c_in:.4f}  dt = {dt:.6f}')
print(f'Time interval: {T:.2f}  ({T * T_ref:.4f} s physical)')

**Define variational problem**

In [ ]:
# Mean velocities for alpha-method time stepping
# alpha_dt = 0.0: explicit Euler, 0.5: trapezoidal, 1.0: implicit Euler
alpha_dt = 0.7

rm  = alpha_dt*r  + (1.0 - alpha_dt)*r0
mm  = alpha_dt*m  + (1.0 - alpha_dt)*m0
Em  = alpha_dt*E  + (1.0 - alpha_dt)*E0

rm1 = alpha_dt*r1 + (1.0 - alpha_dt)*r0
mm1 = alpha_dt*m1 + (1.0 - alpha_dt)*m0
Em1 = alpha_dt*E1 + (1.0 - alpha_dt)*E0

u0_ufl = m0 / (r0 + eps_cav)
u1     = m1 / (r1 + eps_cav)
um1    = alpha_dt*u1 + (1.0 - alpha_dt)*u0_ufl

# Pressure [nondimensional]
p1 = (gamma - 1.0)*(E1 - 0.5*ufl.inner(m1, m1)/(r1 + eps_cav))

# Speed of sound [nondimensional]
c = ufl.sqrt(gamma*(gamma - 1.0)*(E1/r1 - 0.5*ufl.inner(u1, u1)))

# Mach number [nondimensional]
Mach = ufl.sqrt(ufl.inner(u1, u1)) / c

# Outflow pressure: p_fs/p_ref if subsonic, 0 if supersonic
p_out = ufl.conditional(
    ufl.lt(Mach, Constant(msh, PETSc.ScalarType(1.0))),
    Constant(msh, PETSc.ScalarType(float(p_fs / p_ref))),
    Constant(msh, PETSc.ScalarType(0.0)),
)

# Facet normal and local mesh size
normal = ufl.FacetNormal(msh)
h      = ufl.CellDiameter(msh)

# Stabilization parameter (d1 terms commented out as in legacy)
C1    = 1.0
u_mag = ufl.sqrt(ufl.dot(u1, u1))
d1    = C1 / ufl.sqrt((1.0/dt_c)**2 + (u_mag/h)**2)

# Parameters for penalty boundary condition on cylinder
C_alpha       = 1.0e2
alpha_penalty = C_alpha / h

# Residuals for shock-capturing stabilization
res_r = (r1 - r0)/dt_c + ufl.div(u1)*r1 + ufl.inner(ufl.grad(r1), u1)
res_m = (m1 - m0)/dt_c + ufl.div(u1)*m1 + ufl.grad(m1)*u1 + ufl.grad(p1)
res_E = ((E1 - E0)/dt_c + ufl.div(u1)*E1 + ufl.inner(ufl.grad(E1), u1)
         + ufl.div(u1)*p1 + ufl.inner(ufl.grad(p1), u1))

# Smagorinsky model (commented out as in legacy):
#C_t    = 1.0e-2
#nu_turb = C_t*h*h*ufl.sqrt(ufl.inner(ufl.grad(um1), ufl.grad(um1)))

# Shock-capturing stabilization coefficients
C_stab = 1.0
nu_r   = C_stab * h*h * ufl.sqrt(res_r*res_r) / r_in
nu_m   = C_stab * h*h * ufl.sqrt(ufl.inner(res_m, res_m)) / m_in
nu_E   = C_stab * h*h * ufl.sqrt(res_E*res_E) / E_in

# Density variational form
Fr = (ufl.inner((r - r0)/dt_c, d)*ufl.dx
    - ufl.inner(rm*um1, ufl.grad(d))*ufl.dx
    + nu_r * ufl.inner(ufl.grad(rm), ufl.grad(d))*ufl.dx
    + ufl.inner(ufl.dot(um1, normal)*r_in, d)*ds_m(1)
    + ufl.inner(ufl.dot(um1, normal)*rm, d)*ds_m(2))
    #+ d1*inner((r-r0)/dt+div(u1)*rm+inner(grad(rm),u1), dot(grad(d),u1))*dx

# Momentum variational form
#Fm = inner((m-m0)/dt,v)*dx + inner(grad(mm)*um1,v)*dx + inner(div(um1)*mm,v)*dx - p1*div(v)*dx + ...
Fm = (ufl.inner((m - m0)/dt_c, v)*ufl.dx
    - ufl.inner(ufl.outer(mm, um1), ufl.grad(v))*ufl.dx
    - p1*ufl.div(v)*ufl.dx
    + nu_m * ufl.inner(ufl.grad(mm), ufl.grad(v))*ufl.dx
    + alpha_penalty*(ufl.inner(ufl.dot(mm, normal), ufl.dot(v, normal)))*ds_m(5)
    + ufl.dot(u1, normal)*ufl.inner(m_in_vec, v)*ds_m(1)
    + p_out*ufl.dot(normal, v)*ds_m(2)
    + ufl.dot(u1, normal)*ufl.inner(mm, v)*ds_m(2))
    #+ d1*inner((m-m0)/dt+div(u1)*mm+grad(mm)*u1+grad(p1),grad(v)*u1)*dx

# Energy variational form
FE = (ufl.inner((E - E0)/dt_c, q)*ufl.dx
    - ufl.inner(Em*um1, ufl.grad(q))*ufl.dx
    - ufl.inner(p1*um1, ufl.grad(q))*ufl.dx
    + nu_E * ufl.inner(ufl.grad(Em), ufl.grad(q))*ufl.dx
    + ufl.inner(ufl.dot(um1, normal)*E_in, q)*ds_m(1)
    + ufl.inner(ufl.dot(um1, normal)*Em, q)*ds_m(2)
    + ufl.inner(ufl.dot(um1, normal)*p1, q)*ds_m(1)
    + ufl.inner(ufl.dot(um1, normal)*p1, q)*ds_m(2))
    #+ d1*inner((E-E0)/dt+div(u1)*Em+inner(grad(Em),u1)+div(u1)*p1+inner(grad(p1),u1),dot(grad(q),u1))*dx

a_r = form(ufl.lhs(Fr));  L_r = form(ufl.rhs(Fr))
a_m = form(ufl.lhs(Fm));  L_m = form(ufl.rhs(Fm))
a_E = form(ufl.lhs(FE));  L_E = form(ufl.rhs(FE))

**Compute force on boundary**

In [ ]:
# Define the direction of the force to be computed
psi_x = 1.0
psi_y = 0.0

# Build psi: indicator function on cylinder dofs (tag 5), both components
psi    = Function(V, name='psi')
V0c, map_to_V_0 = V.sub(0).collapse()
V1c, map_to_V_1 = V.sub(1).collapse()
psi_0c = Function(V0c); psi_1c = Function(V1c)
cyl_dofs_0 = locate_dofs_topological(V0c, fdim, facet_tags.find(5))
cyl_dofs_1 = locate_dofs_topological(V1c, fdim, facet_tags.find(5))
psi_0c.x.array[cyl_dofs_0] = psi_x
psi_1c.x.array[cyl_dofs_1] = psi_y
psi.x.array[map_to_V_0] = psi_0c.x.array
psi.x.array[map_to_V_1] = psi_1c.x.array
psi.x.scatter_forward()

# Velocity [nondimensional]
u1_psi = m1 / (r1 + eps_cav)

# Pressure [nondimensional]
p1_psi = (gamma - 1.0)*(E1 - 0.5*ufl.inner(m1, m1)/(r1 + eps_cav))

# Compute force through divergence theorem and volume integral
Force_form = form(
    ufl.inner(
        (m1 - m0)/dt_c + ufl.div(u1_psi)*m1 + ufl.grad(m1)*u1_psi,
        psi
    )*ufl.dx
    - p1_psi*ufl.div(psi)*ufl.dx
)

# Compute force through surface integral of pressure
force_normal      = ufl.FacetNormal(msh)
Force_pressure_form = form(ufl.inner(force_normal, psi)*p1_psi*ds_m(5))

# Force normalization to get non-dimensional force coefficient
normalization_cd  = -2.0 / (MD * m_in * u_in)
# Force (volume form) = -F_body (integration by parts, n outward from fluid):
# normalization_cd * F_vol = normalization_cd * (-F_body) > 0 (correct sign for c_D).
# Force_pressure = ∫ p n·ψ ds = +F_body, so normalization_cd * F_prs < 0 (wrong sign).
# Use -normalization_cd so both force coefficients have the same sign.
normalization_cd_p = -normalization_cd
normalization_dim = (MD * L_ref * r_fs * u_fs * u_fs) * 0.5

**Set plotting variables and open export files**

In [ ]:
# --- canonical: ksp_helpers v2 ---
from petsc4py import PETSc


def make_ksp(A, ksp_type="bcgs", pc_type="ilu", amg=False, rtol=1e-6, atol=1e-14, max_it=500):
    ksp = PETSc.KSP().create(A.comm)
    ksp.setOperators(A)
    ksp.setType(ksp_type)
    pc = ksp.getPC()
    if amg:
        pc.setType("hypre")
        pc.setHYPREType("boomeramg")
    else:
        pc.setType(pc_type)
    ksp.setTolerances(rtol=rtol, atol=atol, max_it=max_it)
    ksp.setFromOptions()
    return ksp


def solve_checked(ksp, b, x, name):
    ksp.solve(b, x)
    reason = ksp.getConvergedReason()
    iters = ksp.getIterationNumber()
    if reason < 0:
        raise RuntimeError(
            f"KSP '{name}' diverged: reason={reason}, iterations={iters}"
        )


def solve_checked_snes(snes, name):
    reason = snes.getConvergedReason()
    if reason <= 0:
        iters = snes.getIterationNumber()
        raise RuntimeError(
            f"SNES '{name}' did not converge: reason={reason}, iterations={iters}"
        )
# --- end canonical: ksp_helpers ---

In [ ]:
# Matrices and KSP solvers
A_r = create_matrix(a_r); b_r = create_vector(R)
A_m = create_matrix(a_m); b_m = create_vector(V)
A_E = create_matrix(a_E); b_E = create_vector(Q)

ksp_r = make_ksp(A_r); ksp_m = make_ksp(A_m); ksp_E = make_ksp(A_E)

# Initial conditions
r0.interpolate(lambda x: np.full(x.shape[1], r_in))
m0.interpolate(lambda x: np.vstack([np.full(x.shape[1], m_in), np.zeros(x.shape[1])]))
E0.interpolate(lambda x: np.full(x.shape[1], E_in))
set_bc(r0.x.petsc_vec, bcr); r0.x.scatter_forward()
set_bc(m0.x.petsc_vec, bcm); m0.x.scatter_forward()
set_bc(E0.x.petsc_vec, bcE); E0.x.scatter_forward()
r1.x.array[:] = r0.x.array; r1.x.scatter_forward()
m1.x.array[:] = m0.x.array; m1.x.scatter_forward()
E1.x.array[:] = E0.x.array; E1.x.scatter_forward()

# Open export file
xdmf = XDMFSeries('results-Euler/solution.xdmf')

# Plot frequency and force sampling setup
plot_time            = 0.0
force_array          = []
force_array_pressure = []
time_array           = []
start_sample_time    = 1.0

# Plot initial conditions
plot_scalar(r0, title='Density [nondimensional]')
plot_vector(m0, title='Momentum [nondimensional]')
plot_scalar(E0, title='Total energy [nondimensional]')

In [ ]:
# Triple-decomposition setup (velocity-gradient Schur decomposition)
import scipy.linalg.lapack as la


def triple_decomposition(grad_u):
    new_grad = np.zeros((3, 3))
    for i in range(2):
        for j in range(2):
            new_grad[i, j] = grad_u[i, j]
    def dselect(arg1, arg2): return (arg2 == 0)
    T = la.dgees(dselect, new_grad, sort_t=1)[0]
    sh = np.linalg.norm([T[0, 1], T[0, 2], T[1, 2] + T[2, 1]])
    el = np.linalg.norm(np.diag(T))
    rr = np.sqrt(2 * min(abs(T[1, 2]), abs(T[2, 1]))**2)
    return sh, el, rr


# L2 projection setup: mass matrix assembled once; RHS re-assembled each plot step.
_V2 = functionspace(msh, ("Lagrange", 1))
_u_s = ufl.TrialFunction(_V2); _v_s = ufl.TestFunction(_V2)
_a_proj = form(_u_s * _v_s * ufl.dx)
_A_proj = create_matrix(_a_proj); assemble_matrix(_A_proj, _a_proj); _A_proj.assemble()
_ksp_proj = PETSc.KSP().create(msh.comm)
_ksp_proj.setOperators(_A_proj); _ksp_proj.setType("cg")
_ksp_proj.getPC().setType("icc")
_ksp_proj.setTolerances(rtol=1e-10); _ksp_proj.setFromOptions()

_G00 = Function(_V2); _G01 = Function(_V2)
_G10 = Function(_V2); _G11 = Function(_V2)
_shear      = Function(_V2, name="shear")
_elongation = Function(_V2, name="elongation")
_rotation   = Function(_V2, name="rotation")
_divu       = Function(_V2, name="divu")


def _project_comp(expr, G):
    b = create_vector(_V2)
    with b.localForm() as loc: loc.set(0.0)
    assemble_vector(b, form(expr * _v_s * ufl.dx))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    solve_checked(_ksp_proj, b, G.x.petsc_vec, "_ksp_proj")
    G.x.scatter_forward()


def _do_triple_decomposition():
    u1_vel = m1 / (r1 + eps_cav)
    nG = ufl.nabla_grad(u1_vel)
    _project_comp(nG[0, 0], _G00); _project_comp(nG[0, 1], _G01)
    _project_comp(nG[1, 0], _G10); _project_comp(nG[1, 1], _G11)
    g00 = _G00.x.array.real; g01 = _G01.x.array.real
    g10 = _G10.x.array.real; g11 = _G11.x.array.real
    n_v = _V2.dofmap.index_map.size_local
    for v in range(n_v):
        gm = np.array([[g00[v], g01[v]], [g10[v], g11[v]]])
        sh, el, rr = triple_decomposition(gm)
        _shear.x.array[v] = sh; _elongation.x.array[v] = el; _rotation.x.array[v] = rr
    _divu.x.array[:n_v] = g00[:n_v] + g11[:n_v]

**Time stepping algorithm**

In [ ]:
t_wall0 = time.time()
t    = dt
step = 0
while t < T + 1e-10:

    # Solve non-linear problem
    k = 0
    while k < num_nnlin_iter:

        # Assemble and solve density
        A_r.zeroEntries()
        assemble_matrix(A_r, a_r, bcs=bcr); A_r.assemble()
        with b_r.localForm() as loc: loc.set(0.0)
        assemble_vector(b_r, L_r)
        apply_lifting(b_r, [a_r], bcs=[bcr])
        b_r.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_r, bcr)
        ksp_r.setOperators(A_r, A_r); solve_checked(ksp_r, b_r, r1.x.petsc_vec, "ksp_r")
        r1.x.scatter_forward()
        set_bc(r1.x.petsc_vec, bcr); r1.x.scatter_forward()

        # Assemble and solve momentum
        A_m.zeroEntries()
        assemble_matrix(A_m, a_m, bcs=bcm); A_m.assemble()
        with b_m.localForm() as loc: loc.set(0.0)
        assemble_vector(b_m, L_m)
        apply_lifting(b_m, [a_m], bcs=[bcm])
        b_m.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_m, bcm)
        ksp_m.setOperators(A_m, A_m); solve_checked(ksp_m, b_m, m1.x.petsc_vec, "ksp_m")
        m1.x.scatter_forward()
        set_bc(m1.x.petsc_vec, bcm); m1.x.scatter_forward()

        # Slip boundary condition (commented out as in legacy):
        #bcm_slip_0 = m1[0] - dot(m1, node_normal)*node_normal[0]
        #bcm_slip_1 = m1[1] - dot(m1, node_normal)*node_normal[1]
        #[bc.apply(m1.vector()) for bc in bcm_slip]

        # Assemble and solve total energy
        A_E.zeroEntries()
        assemble_matrix(A_E, a_E, bcs=bcE); A_E.assemble()
        with b_E.localForm() as loc: loc.set(0.0)
        assemble_vector(b_E, L_E)
        apply_lifting(b_E, [a_E], bcs=[bcE])
        b_E.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_E, bcE)
        ksp_E.setOperators(A_E, A_E); solve_checked(ksp_E, b_E, E1.x.petsc_vec, "ksp_E")
        E1.x.scatter_forward()
        set_bc(E1.x.petsc_vec, bcE); E1.x.scatter_forward()

        k += 1

    # Force (sampled once per step after nonlinear convergence)
    F_vol = float(msh.comm.allreduce(assemble_scalar(Force_form), op=MPI.SUM))
    F_prs = float(msh.comm.allreduce(assemble_scalar(Force_pressure_form), op=MPI.SUM))
    if t > start_sample_time:
        force_array.append(normalization_cd * F_vol)
        force_array_pressure.append(normalization_cd_p * F_prs)
        time_array.append(t)

    if t > plot_time:
        print(f'Time [nondimensional] t = {t:.4f}')
        print(f'Time [seconds] t = {t * T_ref:.4f}')

        xdmf.write([r1, m1, E1], t)

        # Derived quantities for plotting
        u1_fn = Function(V, name='Velocity')
        u1_fn.interpolate(Expression(
            m1 / (r1 + eps_cav), V.element.interpolation_points
        ))
        p1_fn = Function(Q, name='Pressure')
        p1_fn.interpolate(Expression(
            (gamma - 1.0)*(E1 - 0.5*ufl.inner(m1, m1)/(r1 + eps_cav)),
            Q.element.interpolation_points
        ))
        e_int = E1/(r1 + eps_cav) - 0.5*ufl.inner(m1/(r1+eps_cav), m1/(r1+eps_cav))
        c_sq  = gamma*(gamma - 1.0)*ufl.max_value(
            e_int, Constant(msh, PETSc.ScalarType(eps_cav))
        )
        Mach_fn = Function(Q, name='Mach')
        Mach_fn.interpolate(Expression(
            ufl.sqrt(ufl.inner(m1/(r1+eps_cav), m1/(r1+eps_cav)))
            / (ufl.sqrt(c_sq) + eps_cav),
            Q.element.interpolation_points
        ))

        plot_scalar(r1,    title='Density [nondimensional]')
        r1_dim = Function(R, name='Density_dim')
        r1_dim.x.array[:] = r1.x.array * r_ref
        plot_scalar(r1_dim, title='Density [kg/m3]')
        plot_vector(m1,    title='Momentum [nondimensional]')
        plot_scalar(E1,    title='Total energy [nondimensional]')
        plot_vector(u1_fn, title='Velocity [nondimensional]')
        plot_scalar(p1_fn, title='Pressure [nondimensional]')
        plot_scalar(Mach_fn, title='Mach number [nondimensional]')

        if time_array:
            plt.figure()
            plt.title(
                f'Force coefficient [nondimensional] '
                f'– scale {normalization_dim:.2f} N/m '
                f'– time scale {T_ref:.4f} s'
            )
            plt.plot(time_array, force_array)
            plt.show()

            plt.figure()
            plt.title(
                f'Pressure force coefficient [nondimensional] '
                f'– scale {normalization_dim:.2f} N/m'
            )
            plt.plot(time_array, force_array_pressure)
            plt.show()


        _do_triple_decomposition()
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for _ax, _func, _title in zip(axes,
                [_shear, _elongation, _rotation],
                ["Shear", "Elongation", "Rotation"]):
            plot_scalar(_func, title=_title, ax=_ax)
        plt.tight_layout(); plt.show()

        fig_divu, ax_divu = plt.subplots(figsize=(8, 3))
        plot_scalar(_divu, title='div u [nondimensional]', ax=ax_divu)
        plt.tight_layout(); plt.show()

        plot_time += T / plot_freq

    # Update solution
    r0.x.array[:] = r1.x.array; r0.x.scatter_forward()
    m0.x.array[:] = m1.x.array; m0.x.scatter_forward()
    E0.x.array[:] = E1.x.array; E0.x.scatter_forward()
    t    += dt
    step += 1

xdmf.close()
t_wall = time.time() - t_wall0
print(f'Wall time: {t_wall:.1f} s')

In compressible flow the velocity-gradient triple decomposition separates the local flow into shear, elongation, and rotation components; for compressible Euler the elongation norm includes the dilatational (volumetric) part, so high elongation coincides with compression waves and shocks. The divergence field $\nabla\cdot u$ directly highlights shock locations where $\nabla\cdot u \ll 0$.

In [ ]:
norm_r1 = float(msh.comm.allreduce(
    assemble_scalar(form(r1*r1*ufl.dx)), op=MPI.SUM))**0.5
norm_m1 = float(msh.comm.allreduce(
    assemble_scalar(form(ufl.inner(m1, m1)*ufl.dx)), op=MPI.SUM))**0.5
norm_E1 = float(msh.comm.allreduce(
    assemble_scalar(form(E1*E1*ufl.dx)), op=MPI.SUM))**0.5
min_r1  = float(msh.comm.allreduce(r1.x.array.min(), op=MPI.MIN))
print(f'||r1|| = {norm_r1:.6f}')
print(f'||m1|| = {norm_m1:.6f}')
print(f'||E1|| = {norm_E1:.6f}')
print(f'min r1 = {min_r1:.6f}')
print(f'cells  = {n_cells}')
print(f'dt     = {dt:.6f}')
print(f'steps  = {step}')
print(f"QOI norm_rho = {norm_r1:.6g}")
print(f"QOI norm_mom = {norm_m1:.6g}")
print(f"QOI norm_E = {norm_E1:.6g}")
print(f"QOI min_rho = {min_r1:.6g}")

# **Discussion**

A stabilized finite element method was implemented in FEniCSx to solve the Euler equations for compressible flow. The shock-capturing GLS stabilization with coefficients $\nu_r$, $\nu_m$, $\nu_E$ provides dissipation proportional to the local residual, localizing artificial diffusion to regions of strong gradients (shocks, contact discontinuities). The $\alpha$-method time stepping ($\alpha = 0.7$) gives an implicitly-biased scheme that improves stability for supersonic flows. The penalty term $\alpha_{\text{penalty}} = C_\alpha/h$ on the cylinder boundary enforces the normal momentum condition in a weak sense.

The ~5\% difference between the volume-integral and pressure-surface-integral force coefficients is **UNEXPLAINED**. Candidates include discretization error in the stabilized volume form and shock-capturing stabilization terms that appear in the volume residual but are absent from the pressure surface integral.

**Legacy-diff table — force-coefficient computation:**

| item | legacy (`johanhoffman/DD2365`) | port (`DD2365_FEniCSx`) | notes |
|---|---|---|---|
| `force_array_pressure` content | appended `normalization_cd*F` (volume force) — duplicated the volume-force plot | appends `normalization_cd_p*F_prs` (pressure-surface integral) | **legacy bug**: `normalization_cd*F` used `F` (volume form) instead of `F_pressure`; the legacy pressure-force plot was identical to the volume-force plot |
| pressure-force sign | n/a (wrong content throughout) | `normalization_cd_p = -normalization_cd` | Force (volume) = −F_body → `normalization_cd*F_vol` > 0; Force_pressure = +F_body → must use `−normalization_cd` to get the same positive sign |

| triple decomposition | absent | added | authorized addition: velocity-gradient Schur decomposition + L2 projection onto P1, shear/elongation/rotation + div u plotted at plot times |